In [108]:
from datetime import datetime, timedelta
import pandas as pd
import calendar
from IPython.display import display, HTML

def render_clean_standalone_holiday_calendar(csv_file_path, month, year):
    # 1. Load multi-tier spreadsheet data
    try:
        df = pd.read_csv(csv_file_path)
    except FileNotFoundError:
        print(f"Error: Could not find the file '{csv_file_path}'. Please run your scheduling engine first.")
        return
    
    # Parse dates and map out the global schedule index using exact date matching
    df['Date'] = pd.to_datetime(df['Date']).dt.date
    
    # Automatically harvest unique staff names across the entire spreadsheet dataset
    unique_staff = set()
    shift_map = {}
    
    def clean_cell_value(val):
        if pd.isna(val) or str(val).strip().lower() in ['nan', 'none', '']:
            return "None"
        return str(val).strip()

    # Pre-populate shift data into a global hash map using unique date keys
    for _, row in df.iterrows():
        row_date = row['Date']
        day_of_week_str = str(row['Day Week'] if 'Day Week' in row.index else row['Day of Week'])
        is_holiday = "[HOLIDAY]" in day_of_week_str
        
        shift_map[row_date] = {
            'is_holiday': is_holiday,
            'day_hospital': clean_cell_value(row['Day Shift (12hr)']),
            'night_hospital': clean_cell_value(row['Night Shift (12hr)']),
            'full_clinic': clean_cell_value(row['Full Clinic (8hr)']),
            'half_clinic': clean_cell_value(row['Half Clinic (4hr)'])
        }
        
        # Only parse uniquely filtered personnel list from the explicitly requested month grid
        if row_date.month == month and row_date.year == year:
            for col in ['Day Shift (12hr)', 'Night Shift (12hr)', 'Full Clinic (8hr)', 'Half Clinic (4hr)']:
                val_clean = clean_cell_value(row[col])
                if val_clean != "None" and "CLOSED" not in val_clean:
                    for name in val_clean.split(", "):
                        unique_staff.add(name)

    # Instantiate Sunday-start continuous iterable engine
    cal = calendar.Calendar(firstweekday=6)
    grid_dates = list(cal.itermonthdates(year, month))
    month_name = calendar.month_name[month]

    # 2. Build the HTML Dropdown Selector
    dropdown_html = f"""
    <div style="font-family: 'Segoe UI', Arial, sans-serif; margin-bottom: 20px; background-color: #f8fafc; padding: 15px; border-radius: 6px; border: 1px solid #e2e8f0; display: inline-block;">
        <label for="staffFilter" style="font-weight: 600; margin-right: 12px; color: #0f172a; font-size: 14px;">🔎 Filter Calendar by Staff Member:</label>
        <select id="staffFilter" onchange="filterCalendarView(this.value)" style="padding: 8px 16px; border-radius: 4px; border: 1px solid #cbd5e1; background-color: white; color: #0f172a; font-weight: 500; font-size: 14px; outline: none; cursor: pointer; box-shadow: 0 1px 2px rgba(0,0,0,0.05);">
            <option value="ALL">📋 Show Master Schedule (Full Team View)</option>
    """
    for name in sorted(list(unique_staff)):
        dropdown_html += f'<option value="{name}">{name}</option>'
    dropdown_html += "</select></div>"

    # 3. Construct Calendar Grid Layout with Leading/Trailing Styling Changes
    calendar_html = f"""
    <style>
        .cal-table {{ border-collapse: collapse; font-family: 'Segoe UI', Arial, sans-serif; width: 100%; max-width: 1100px; margin-bottom: 20px; box-shadow: 0 5px 15px rgba(0,0,0,0.06); border-radius: 8px; overflow: hidden; }}
        .cal-header {{ background-color: #0f172a; color: white; text-align: center; font-size: 24px; font-weight: bold; padding: 16px; }}
        .cal-day-name {{ background-color: #f1f5f9; color: #475569; text-align: center; font-weight: 600; padding: 12px; width: 14.28%; border: 1px solid #e2e8f0; text-transform: uppercase; font-size: 11px; letter-spacing: 0.5px; }}
        .cal-cell {{ border: 1px solid #e2e8f0; height: 165px; vertical-align: top; padding: 8px; width: 14.28%; background-color: white; transition: background-color 0.2s; }}
        
        /* New explicit structural styles for padding adjacent blocks */
        .cal-padding-day {{ background-color: #f8fafc !important; }}
        .cal-padding-day .day-number {{ color: #94a3b8 !important; }}
        .cal-padding-day .shift-tag {{ opacity: 0.5; filter: grayscale(30%); }}
        
        .day-number {{ font-weight: bold; color: #1e293b; font-size: 14px; margin-bottom: 6px; border-bottom: 1px solid #f1f5f9; padding-bottom: 4px; display: flex; justify-content: space-between; align-items: center; height: 24px;}}
        .holiday-badge {{ font-size: 10px; color: #ef4444; background-color: #fee2e2; padding: 1px 6px; border-radius: 10px; font-weight: 600; letter-spacing: 0.3px; }}
        
        .shift-tag {{ font-size: 11px; padding: 4px 6px; margin: 4px 0; border-radius: 4px; line-height: 1.3; font-weight: 600; display: block; transition: all 0.2s ease; }}
        .tier-day {{ background-color: cyan; color: #0369a1; border-left: 4px solid #0284c7; }}
        .tier-night {{ background-color: navy; color: yellow; border-left: 4px solid #8b5cf6; }}
        .tier-full {{ background-color: #dcfce7; color: #166534; border-left: 4px solid #22c55e; }}
        .tier-half {{ background-color: #fef9c3; color: #854d0e; border-left: 4px solid #eab308; }}
        .tier-holiday-closed {{ background-color: #fef2f2; color: #991b1b; border: 1px dashed #fee2e2; font-weight: 500; text-align: center; font-style: italic; padding: 8px 4px; margin-top: 10px; border-radius: 4px; }}
        
        .js-muted {{ background-color: #f1f5f9 !important; color: #94a3b8 !important; border-left: 4px solid #cbd5e1 !important; opacity: 0.15 !important; font-weight: normal !important; }}
    </style>
    
    <table class="cal-table">
        <tr><th colspan="7" class="cal-header">{month_name} {year}</th></tr>
        <tr>
            <td class="cal-day-name">Sun</td><td class="cal-day-name">Mon</td><td class="cal-day-name">Tue</td>
            <td class="cal-day-name">Wed</td><td class="cal-day-name">Thu</td><td class="cal-day-name">Fri</td>
            <td class="cal-day-name">Sat</td>
        </tr>
    """

    # Group into exact 7-day grid week blocks
    for i in range(0, len(grid_dates), 7):
        week = grid_dates[i:i+7]
        calendar_html += "<tr>"
        
        for date_obj in week:
            is_current_month = (date_obj.month == month and date_obj.year == year)
            cell_class = "cal-cell" if is_current_month else "cal-cell cal-padding-day"
            
            calendar_html += f'<td class="{cell_class}">'
            
            has_data = date_obj in shift_map
            day_is_holiday = has_data and shift_map[date_obj]['is_holiday']
            
            if day_is_holiday:
                calendar_html += f'<div class="day-number">{date_obj.day} <span class="holiday-badge">🎉 HOLIDAY</span></div>'
            else:
                calendar_html += f'<div class="day-number">{date_obj.day}</div>'
            
            if has_data:
                data = shift_map[date_obj]
                
                # 1. Day Hospital Shift Layer
                if data['day_hospital'] != "None":
                    calendar_html += f'<div class="shift-tag tier-day" data-workers="{data["day_hospital"]}">☀️ {data["day_hospital"]}</div>'
                
                # 2. Night Hospital Shift Layer
                if data['night_hospital'] != "None":
                    calendar_html += f'<div class="shift-tag tier-night" data-workers="{data["night_hospital"]}">🌙 {data["night_hospital"]}</div>'
                
                # 3. Full Clinic Layer
                if data['full_clinic'] != "None":
                    if "CLOSED" in data['full_clinic']:
                        calendar_html += f'<div class="tier-holiday-closed">🏥 Clinic Closed</div>'
                    else:
                        for staff in data['full_clinic'].split(", "):
                            calendar_html += f'<div class="shift-tag tier-full" data-workers="{staff}">💼 {staff}</div>'
                
                # 4. Half Clinic Layer
                if data['half_clinic'] != "None" and "CLOSED" not in data['half_clinic']:
                    for staff in data['half_clinic'].split(", "):
                        calendar_html += f'<div class="shift-tag tier-half" data-workers="{staff}">⏱️ {staff} (Half)</div>'
            
            calendar_html += '</td>'
        calendar_html += "</tr>"
        
    calendar_html += "</table>"

    # 4. Inject Interactive JavaScript Filtering Client Engine
    js_script = """
    <script>
    function filterCalendarView(workerName) {
        var tags = document.getElementsByClassName('shift-tag');
        for (var i = 0; i < tags.length; i++) {
            var assignedWorkers = tags[i].getAttribute('data-workers');
            if (workerName === 'ALL') {
                tags[i].classList.remove('js-muted');
            } else {
                if (assignedWorkers && assignedWorkers.split(', ').includes(workerName)) {
                    tags[i].classList.remove('js-muted');
                } else {
                    tags[i].classList.add('js-muted');
                }
            }
        }
    }
    </script>
    """
    # 1. Name the output file
    output_filename = "Master_Hospital_Schedule.html"
    
    # 2. Add an optional wrapper to center the grid on the page nicely
    full_page_markup = f"""<!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>Hospital Roster & Clinic Schedule</title>
        <style>
            body {{ 
                background-color: #f8fafc; 
                padding: 20px; 
                margin: 0;
            }}
            /* Keep things clean when saving or printing directly to a PDF document */
            @media print {{
                body {{ background-color: white; padding: 0; }}
                .no-print {{ display: none !important; }}
            }}
        </style>
    </head>
    <body>
    
        {dropdown_html + calendar_html + js_script}
    
    </body>
    </html>"""
    
    # 3. Write and save the document to your local machine storage
    with open(output_filename, "w", encoding="utf-8") as file:
        file.write(full_page_markup)
    
    print(f"🎉 Success! Static file saved to your workspace as: '{output_filename}'")

    # Print out combined HTML layout frames into the active notebook output display
    display(HTML(dropdown_html + calendar_html + js_script))
render_clean_standalone_holiday_calendar("hospital_holiday_40hr_schedule.csv", month=11, year=2026)


🎉 Success! Static file saved to your workspace as: 'Master_Hospital_Schedule.html'
